# PocketVina Atlas

<a href="https://colab.research.google.com/github/BIMSBbioinfo/PocketVina/blob/main/notebooks/atlas_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The [PocketVina Atlas](https://bimsbstatic.mdc-berlin.de/akalin/asarigun/pocketvina_atlas/) holds docking results
computed in advance with PocketVina: a library of drugs docked against the whole human proteome. This notebook searches
these results and shows the best scores, one protein × drug pair in 3D, its PoseBusters checks and its poses as an SDF
file. At the end it downloads a part of the Atlas for offline use. You run no docking yourself.

No GPU is needed. On Colab: **Runtime → Run all**.

## Install

On Colab this installs PocketVina (`pip install pocketvina`). Elsewhere it installs nothing: install PocketVina
0.3 or later first (`pip install -U pocketvina`).

In [ ]:
import subprocess, sys
from importlib.metadata import version

if "google.colab" in sys.modules:
    loaded = {m: version(m) for m in ("numpy", "pandas", "pyarrow") if m in sys.modules}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pocketvina>=0.3"], check=True)
    if any(version(m) != v for m, v in loaded.items()):
        raise RuntimeError("pip updated numpy, pandas or pyarrow: Runtime > Restart session, then Run all")
import pocketvina

print("PocketVina", pocketvina.__version__)
if not hasattr(pocketvina, "Atlas"):  # an older PocketVina
    raise RuntimeError("the PocketVina Atlas needs PocketVina 0.3 or later: pip install -U pocketvina")

## Open the Atlas

`Atlas()` reads the public Atlas over the internet. Each call reads only the files it needs.

In [ ]:
from pocketvina import Atlas

atlas = Atlas()
atlas

## Search

Proteins and drugs whose identifiers match, best match first.

In [ ]:
atlas.search("imatinib")

## A protein

Gene, name, AlphaFold model and pockets. `protein=` takes any identifier: gene, UniProt accession, entry name,
Entrez, HGNC or Ensembl gene ID.

In [ ]:
atlas.protein("ABL1")

## Best scores

The drugs with the best Vina score on ABL1, then the proteins imatinib scores best on.

In [ ]:
display(atlas.best_scores(protein="ABL1", top=10))
atlas.best_scores(ligand="imatinib", top=10)

## One protein × drug pair

`select` picks the poses of a pair, as for your own docking results: here the best pose in every pocket. `show()`
draws the protein's page with this pair selected.

In [ ]:
pair = atlas.select(protein="ABL1", ligand="imatinib", pocket="all")
pair.show()

## PoseBusters

Checks that the selected poses are physically plausible (bond lengths, clashes, ...).

In [ ]:
pair.bust()

## Save

The poses as an SDF file and the protein (its AlphaFold model) as a PDB file. On Colab they are in the *Files*
panel on the left.

In [ ]:
print(pair.export())
print(pair.export_protein())

## Work offline

`download` copies what the Atlas has on some proteins into a folder. The pose files are large: it says the size
first. `Atlas("my_atlas")` then reads the folder without internet.

In [ ]:
atlas.download(proteins=["ABL1"], out="my_atlas")
offline = Atlas("my_atlas")
offline.best_scores(protein="ABL1", top=5)

## Next

- The [PocketVina Atlas page](https://bioinformatics.mdc-berlin.de/PocketVina/atlas.html) of the documentation:
  identifiers, the command line and offline use.
- The [Atlas website](https://bimsbstatic.mdc-berlin.de/akalin/asarigun/pocketvina_atlas/) describes the data.
- To dock your own proteins and ligands, see the
  [quickstart notebook](https://colab.research.google.com/github/BIMSBbioinfo/PocketVina/blob/main/notebooks/quickstart_colab.ipynb).